<a href="https://colab.research.google.com/github/aeverson28/data-sanitization-pipeline/blob/main/mini_projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**1 - Validação e Tratamento de Dados Ausentes:**

Criar uma lógica iterativa para ler o arquivo de produtos. Sempre que encontrar um valor **nulo/vazio** na coluna *product_category_name,* ele deve ser preenchido com a string **"Sem Categoria"**. Para os valores nulos nas dimensões físicas (*product_weight_g, product_length_cm, etc.*), você deve criar uma regra de corte (ex: descartar o registro ou atribuir um valor numérico padrão, como média por exemplo), justificando a sua escolha técnica nos comentários do código.

In [23]:
import csv

def sanitizar_dataset_produtos(caminho_entrada, caminho_saida):
    """
    Lê um arquivo CSV de produtos, preenche categorias vazias
    e salva o resultado em um novo arquivo.
    """
    print("Iniciando o pipeline de sanitização")

    # 1. Abrimos ambos os arquivos no mesmo bloco 'with' para garantir o fechamento seguro
    with open(caminho_entrada, mode='r', encoding='utf-8') as arquivo_entrada, \
         open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_saida:

        # 2. Criamos o Leitor
        entrada_produtos_csv = csv.DictReader(arquivo_entrada, delimiter=",")

        # 3. Criamos o Escritor a partir das colunas originais
        colunas = entrada_produtos_csv.fieldnames
        saida_produtos_csv = csv.DictWriter(arquivo_saida, fieldnames=colunas, delimiter=",")

        # 4. Escrevemos a primeira linha (cabeçalho) no novo arquivo
        saida_produtos_csv.writeheader()

        # Contadores para acompanharmos o processamento
        linhas_processadas = 0
        categorias_vazias_corrigidas = 0

        # 5. Iteramos linha a linha (cada linha é um dicionário)
        for linha in entrada_produtos_csv:
            linhas_processadas += 1

            # Pegamos o valor atual da categoria (se não existir a coluna, retorna string vazia)
            categoria_atual = linha.get("product_category_name", "")

            # 6. Lógica if/else para tratamento de nulos/vazios
            if not categoria_atual or not categoria_atual.strip():
                linha["product_category_name"] = "Sem categoria"
                categorias_vazias_corrigidas += 1
            else:
                linha["product_category_name"] = categoria_atual.strip()

            # 7. Salvamos a linha processada no arquivo de saída
            saida_produtos_csv.writerow(linha)

    # Resumo da Sanitização
    print("--- Sanitização Concluída com Sucesso ---")
    print(f"Total de registros processados: {linhas_processadas}")
    print(f"Categorias vazias preenchidas: {categorias_vazias_corrigidas}")
    print(f"Arquivo salvo em: {caminho_saida}")


caminho_arquivo_produtos = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos.csv"
caminho_arquivo_saida = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_sanitizado.csv"

sanitizar_dataset_produtos(caminho_arquivo_produtos, caminho_arquivo_saida)

Iniciando o pipeline de sanitização
--- Sanitização Concluída com Sucesso ---
Total de registros processados: 32951
Categorias vazias preenchidas: 610
Arquivo salvo em: /content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_sanitizado.csv


Verificação da quantidade de valores nulos/vazios das colunas de dimensões físicas, dessa forma será possivel uma melhor análise de qual a melhor decisão: descartar as linhas com valores nulos/vazios ou preenche-las com a média ou a mediana.

In [24]:
def verificar_nulos_dimensoes(caminho_arquivo):
    print("Iniciando a verificação de valores nulos\n")

    # Lista com o nome das colunas
    colunas_alvo = [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]

    # Usando um dicionário para guardar a contagem de cada coluna
    contagem_nulos = {coluna: 0 for coluna in colunas_alvo}
    total_linhas = 0

    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        leitor_csv = csv.DictReader(arquivo, delimiter=",")

        for linha in leitor_csv:
            total_linhas += 1

            # Verificamos cada uma das 4 colunas dentro da mesma linha
            for coluna in colunas_alvo:
                # Pegamos o valor, se a coluna não existir pegamos vazio ("")
                valor = linha.get(coluna, "")

                # Se for vazio ou contiver apenas espaços, somamos no contador
                if not valor or not valor.strip():
                    contagem_nulos[coluna] += 1

    # ==========================================
    # Exibição do Relatório
    # ==========================================
    print("==============================================")
    print("      RELATÓRIO DE VALORES NULOS/VAZIOS")
    print("==============================================")
    print(f"Total de produtos analisados: {total_linhas}\n")

    for coluna, quantidade in contagem_nulos.items():
        print(f"[{coluna}]: {quantidade} linhas vazias")
    print("==============================================\n")

# ==========================================
# Executando a verificação
# ==========================================
# Vamos ler do arquivo que já limpamos a categoria no passo anterior
caminho_arquivo_sanitizado = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_sanitizado.csv"

verificar_nulos_dimensoes(caminho_arquivo_sanitizado)

Iniciando a verificação de valores nulos

      RELATÓRIO DE VALORES NULOS/VAZIOS
Total de produtos analisados: 32951

[product_weight_g]: 2 linhas vazias
[product_length_cm]: 2 linhas vazias
[product_height_cm]: 2 linhas vazias
[product_width_cm]: 2 linhas vazias



Como temos apenas duas linhas vazias, o descarte dessas linhas não comprometerá a análise posterior.

Descarte dos registros com valores nulos/vazios:

In [25]:
def remover_produtos_sem_dimensoes(caminho_entrada, caminho_saida):
    print("Iniciando a remoção de registros nulos\n")

    colunas_alvo = [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]

    # Abrindo os arquivos
    with open(caminho_entrada, mode='r', encoding='utf-8') as arquivo_entrada, \
         open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_saida:

        leitor_csv = csv.DictReader(arquivo_entrada, delimiter=",")

        # Preparando o escritor com as mesmas colunas
        colunas = leitor_csv.fieldnames
        escritor_csv = csv.DictWriter(arquivo_saida, fieldnames=colunas, delimiter=",")
        escritor_csv.writeheader()

        linhas_mantidas = 0
        linhas_descartadas = 0

        # Iterando sobre os dados
        for linha in leitor_csv:
            descartar_linha = False

            # Verificamos as 4 colunas para a linha atual
            for coluna in colunas_alvo:
                valor = linha.get(coluna, "")

                # Se encontrar vazio em QUALQUER uma das colunas, marca para descarte e para a checagem
                if not valor or not valor.strip():
                    descartar_linha = True
                    break

            # Se a flag de descarte for True ignoramos a linha.
            # Se for False, nós a salvamos no novo arquivo.
            if descartar_linha:
                linhas_descartadas += 1
            else:
                escritor_csv.writerow(linha)
                linhas_mantidas += 1

    print("==============================================")
    print("           RELATÓRIO DE LIMPEZA")
    print("==============================================")
    print(f"Linhas preservadas e salvas: {linhas_mantidas}")
    print(f"Linhas DESCARTADAS:          {linhas_descartadas}")
    print(f"Arquivo final salvo em:      {caminho_saida}")
    print("==============================================\n")

# ==========================================
# Execução do Código
# ==========================================
# Usamos o arquivo de entrada da etapa anterior e geramos um novo arquivo final
caminho_arquivo_entrada = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_sanitizado.csv"
caminho_arquivo_saida = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_limpo.csv"

remover_produtos_sem_dimensoes(caminho_arquivo_entrada, caminho_arquivo_saida)

Iniciando a remoção de registros nulos

           RELATÓRIO DE LIMPEZA
Linhas preservadas e salvas: 32949
Linhas DESCARTADAS:          2
Arquivo final salvo em:      /content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_limpo.csv



**2 - Padronização de Strings e Regex:**

Garantir que todos os nomes de categorias de produtos sejam convertidos estritamente para **letras minúsculas** *(.lower())* e aplicar a **remoção de espaços em branco excedentes no início e no fim das strings** *(.strip())*. Além disso, utilizar Expressões Regulares *(módulo re)* para **limpar eventuais caracteres especiais ou pontuações indevidas** dos nomes das categorias.

In [26]:
import re  # Importando o módulo nativo de Expressões Regulares

def padronizar_categorias_completo(caminho_entrada, caminho_saida):
    print("Iniciando a padronização das categorias\n")

    # Pré-compilamos a expressão regular para o código rodar mais rápido
    # Essa regra apaga tudo que NÃO seja letra, número, underline (_), espaço ou hífen
    padrao_regex = re.compile(r'[^\w\s-]')

    with open(caminho_entrada, mode='r', encoding='utf-8') as arquivo_entrada, \
         open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_saida:

        leitor_csv = csv.DictReader(arquivo_entrada, delimiter=",")

        colunas = leitor_csv.fieldnames
        escritor_csv = csv.DictWriter(arquivo_saida, fieldnames=colunas, delimiter=",")
        escritor_csv.writeheader()

        linhas_processadas = 0

        for linha in leitor_csv:
            categoria_atual = linha.get("product_category_name", "")

            # Passo 1: Limpar os caracteres especiais com Regex
            categoria_limpa = padrao_regex.sub('', categoria_atual)

            # Passo 2: Remover espaços excedentes nas pontas e converter para minúsculas
            linha["product_category_name"] = categoria_limpa.strip().lower()

            # Salvar no novo arquivo
            escritor_csv.writerow(linha)
            linhas_processadas += 1

    # ==========================================
    # Relatório Final
    # ==========================================
    print("==============================================")
    print("           RELATÓRIO DE PADRONIZAÇÃO")
    print("==============================================")
    print(f"Total de linhas processadas: {linhas_processadas}")
    print("Todas as categorias foram higienizadas com Regex, formatadas sem espaços extras e em minúsculas.")
    print(f"Arquivo salvo em: {caminho_saida}")
    print("==============================================\n")

# ==========================================
# Execução do Código
# ==========================================
# Puxando o arquivo gerado na tarefa 2 (limpo de nulos) e aplicando a nova regra completa
caminho_arquivo_limpo = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_limpo.csv"
caminho_arquivo_padronizado = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_padronizado.csv"

padronizar_categorias_completo(caminho_arquivo_limpo, caminho_arquivo_padronizado)

Iniciando a padronização das categorias

           RELATÓRIO DE PADRONIZAÇÃO
Total de linhas processadas: 32949
Todas as categorias foram higienizadas com Regex, formatadas sem espaços extras e em minúsculas.
Arquivo salvo em: /content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos_padronizado.csv



**3 - Lógica de Regra de Negócio (Filtros e Validação):**


Verificar a seguinte hipótese: as datas de entrega (order_delivered_customer_date) estão vazias devido ao status do pedido (order_status) estar cancelado


In [27]:
def testar_hipotese_cancelados(caminho_arquivo):
    print("Iniciando o teste de hipótese\n")

    total_cancelados = 0
    cancelados_sem_data = 0
    cancelados_com_data = 0

    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        leitor_csv = csv.DictReader(arquivo, delimiter=",")

        for linha in leitor_csv:
            # Pegamos o status e a data de entrega
            status = linha.get("order_status", "").strip().lower()
            data_entrega = linha.get("order_delivered_customer_date", "").strip()

            # Filtramos apenas os pedidos cancelados
            if status == "canceled":
                total_cancelados += 1

                # Verificamos se a data está vazia
                if not data_entrega:
                    cancelados_sem_data += 1
                else:
                    cancelados_com_data += 1

    # ==========================================
    # Relatório Final
    # ==========================================
    print("==============================================")
    print("           RESULTADO DA HIPÓTESE")
    print("==============================================")
    print(f"Total de pedidos 'canceled':      {total_cancelados}")
    print(f"Cancelados COM a data VAZIA:      {cancelados_sem_data}")
    print(f"Cancelados COM a data PREENCHIDA: {cancelados_com_data}  <-- Anomalias!")
    print("==============================================\n")

    # Conclusão lógica
    if cancelados_com_data == 0:
        print("Todos os pedidos cancelados não possuem data de entrega.")
    else:
        print(f"Encontramos {cancelados_com_data} pedidos cancelados que possuem data de entrega.")

# ==========================================
# Execução do Código
# ==========================================
caminho_arquivo_pedidos = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_pedidos.csv"

testar_hipotese_cancelados(caminho_arquivo_pedidos)

Iniciando o teste de hipótese

           RESULTADO DA HIPÓTESE
Total de pedidos 'canceled':      625
Cancelados COM a data VAZIA:      619
Cancelados COM a data PREENCHIDA: 6  <-- Anomalias!

Encontramos 6 pedidos cancelados que possuem data de entrega.


**4 - Formatação Temporal (Datetime):**

Utilize o módulo nativo datetime para ler a coluna de data de aprovação do pedido (order_approved_at) e convertê-la do formato de string original (ex: "2017-05-16 15:05:35") para um formato de data simplificado brasileiro (ex: "16/05/2017").


In [28]:
from datetime import datetime

def formatar_data_sem_try(caminho_entrada, caminho_saida):
    print("Iniciando a conversão de datas\n")

    # Criamos um "detetor de formato".
    # \d{4} significa "exatamente 4 números", \d{2} significa "exatamente 2 números"
    # O ^ indica o começo do texto e o $ o final.
    padrao_formato_data = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")

    linhas_processadas = 0
    datas_convertidas = 0
    datas_vazias = 0
    datas_ignoradas = 0

    with open(caminho_entrada, mode='r', encoding='utf-8') as arquivo_entrada, \
         open(caminho_saida, mode='w', encoding='utf-8', newline='') as arquivo_saida:

        leitor_csv = csv.DictReader(arquivo_entrada, delimiter=",")
        colunas = leitor_csv.fieldnames
        escritor_csv = csv.DictWriter(arquivo_saida, fieldnames=colunas, delimiter=",")
        escritor_csv.writeheader()

        for linha in leitor_csv:
            linhas_processadas += 1
            data_original_str = linha.get("order_approved_at", "").strip()

            # 1. Verifica se a string não está vazia
            if data_original_str:

                # 2. "Olhe antes de saltar": Verifica se o texto é idêntico ao formato esperado
                if padrao_formato_data.match(data_original_str):
                    # Como já validámos que o texto é seguro, a conversão não falhará
                    data_obj = datetime.strptime(data_original_str, "%Y-%m-%d %H:%M:%S")
                    linha["order_approved_at"] = data_obj.strftime("%d/%m/%Y")
                    datas_convertidas += 1
                else:
                    # Se não passar no teste do Regex, não tentamos converter para não dar erro
                    print(f"Aviso: Dado com formato anómalo na linha {linhas_processadas}: {data_original_str}")
                    datas_ignoradas += 1

            else:
                datas_vazias += 1

            # Escreve a linha no novo ficheiro
            escritor_csv.writerow(linha)

    # ==========================================
    # Relatório Final
    # ==========================================
    print("==============================================")
    print("           RELATÓRIO DE DATAS")
    print("==============================================")
    print(f"Total de pedidos processados: {linhas_processadas}")
    print(f"Datas convertidas (DD/MM/YYYY): {datas_convertidas}")
    print(f"Datas vazias (ignoradas):       {datas_vazias}")
    print(f"Anomalias de formato (ignoradas): {datas_ignoradas}")
    print("==============================================\n")

# ==========================================
# Execução do Código
# ==========================================
caminho_arquivo_pedidos = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_pedidos.csv"
caminho_arquivo_pedidos_datas = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_pedidos_datas_formatadas.csv"

formatar_data_sem_try(caminho_arquivo_pedidos, caminho_arquivo_pedidos_datas)

Iniciando a conversão de datas

           RELATÓRIO DE DATAS
Total de pedidos processados: 99441
Datas convertidas (DD/MM/YYYY): 99281
Datas vazias (ignoradas):       160
Anomalias de formato (ignoradas): 0



**5 - Relatório de Status Manual:**

Ao final do processamento, seu script deve calcular e exibir na tela um sumário estatístico construído manualmente (**contagem total de linhas processadas, total de registros nulos corrigidos e total de pedidos cancelados identificados**), validando se a base está sanitizada.

In [29]:
def executar_relatorio():
    # Os 3 contadores do relatório
    contagem_total_linhas_processadas = 0
    total_registros_nulos_corrigidos = 0
    total_pedidos_cancelados_identificados = 0

    # ==========================================
    # 1. PROCESSANDO DATASET DE PRODUTOS
    # ==========================================
    caminho_prod_in = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_produtos.csv"
    caminho_prod_out = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto.csv"

    colunas_dimensoes = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
    padrao_regex_prod = re.compile(r'[^\w\s-]')

    with open(caminho_prod_in, mode='r', encoding='utf-8') as fin, \
         open(caminho_prod_out, mode='w', encoding='utf-8', newline='') as fout:

        leitor = csv.DictReader(fin)
        escritor = csv.DictWriter(fout, fieldnames=leitor.fieldnames)
        escritor.writeheader()

        for linha in leitor:
            contagem_total_linhas_processadas += 1

            # Descartar se tiver dimensão nula
            descartar = False
            for col in colunas_dimensoes:
                val = linha.get(col, "")
                if not val or not val.strip():
                    descartar = True
                    break
            if descartar:
                continue

            # Limpeza da Categoria (AQUI nós corrigimos nulos e contabilizamos)
            cat = linha.get("product_category_name", "")
            if not cat or not cat.strip():
                linha["product_category_name"] = "sem categoria"
                total_registros_nulos_corrigidos += 1  # Soma apenas os nulos corrigidos aqui
            else:
                cat_limpa = padrao_regex_prod.sub('', cat)
                linha["product_category_name"] = cat_limpa.strip().lower()

            escritor.writerow(linha)

    # ==========================================
    # 2. PROCESSANDO DATASET DE PEDIDOS
    # ==========================================
    caminho_ped_in = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_pedidos.csv"
    caminho_ped_out = "/content/drive/MyDrive/Colab Notebooks/Mini_projeto/dataset_pedidos_final_sanitizado.csv"

    padrao_data = re.compile(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$")

    with open(caminho_ped_in, mode='r', encoding='utf-8') as fin, \
         open(caminho_ped_out, mode='w', encoding='utf-8', newline='') as fout:

        leitor = csv.DictReader(fin)
        escritor = csv.DictWriter(fout, fieldnames=leitor.fieldnames)
        escritor.writeheader()

        for linha in leitor:
            contagem_total_linhas_processadas += 1

            # Contagem de cancelados
            status = linha.get("order_status", "").strip().lower()
            if status == "canceled":
                total_pedidos_cancelados_identificados += 1

            # Tratamento de Data (As vazias permanecem nulas, SEM somar no contador)
            dt_aprov = linha.get("order_approved_at", "").strip()

            if dt_aprov and padrao_data.match(dt_aprov):
                # Só converte se existir e for uma data válida
                dt_obj = datetime.strptime(dt_aprov, "%Y-%m-%d %H:%M:%S")
                linha["order_approved_at"] = dt_obj.strftime("%d/%m/%Y")

            escritor.writerow(linha)

    # ==========================================
    # RELATÓRIO FINAL ESTATÍSTICO
    # ==========================================
    print(f"contagem total de linhas processadas: {contagem_total_linhas_processadas}")
    print(f"total de registros nulos corrigidos: {total_registros_nulos_corrigidos}")
    print(f"total de pedidos cancelados identificados: {total_pedidos_cancelados_identificados}")

# ==========================================
# Execução
# ==========================================
executar_relatorio()

contagem total de linhas processadas: 132392
total de registros nulos corrigidos: 609
total de pedidos cancelados identificados: 625
